In [ ]:
import pandas as pd
import openpyxl

In [ ]:
df = pd.read_excel("data/asset_data.xlsx")

In [ ]:
df["CURRENCY"].value_counts()

In [ ]:
df = df.drop(columns={"CURRENCY"})

In [ ]:
print(df.head(10))

In [ ]:
df["ID"].value_counts()

In [ ]:
df["DATE"] = pd.to_datetime(df["DATE"])
df_pivot = df.pivot_table(index="DATE", columns="ID", values="PX_LAST", aggfunc="mean")

In [ ]:
df_pivot = df_pivot.reset_index()

In [ ]:
df_pivot.isna().sum()

In [ ]:
df_filtered = df_pivot[df_pivot["DATE"] >= "1994-02-27"]
df_filtered

In [ ]:
df_filtered.isna().sum()

In [ ]:
df_selected = df_filtered.drop(columns={"BISHUSI Index", "HPI LEVL Index", "LF98TRUU Index", "LUACTRUU Index", "M1US000V Index", "M1US000G Index", "WBCOCBRN Index"})

In [ ]:
df_selected

In [ ]:
import yfinance as yf

# Download S&P 500 data for the given dates in df_selected
sp500 = yf.download("^GSPC", start=df_selected["DATE"].min(), end=df_selected["DATE"].max() + pd.Timedelta(days=1))

# Reset index to have 'DATE' as a column
sp500 = sp500.reset_index()

# Filter only the dates present in df_selected
sp500_filtered = sp500[sp500["Date"].isin(df_selected["DATE"])][["Date", "Close"]].rename(columns={"Date": "DATE", "Close": "SP500_Close"})

print(sp500_filtered)

In [ ]:
# Flatten columns in sp500_filtered if they are multi-indexed
sp500_filtered.columns = [col[0] if isinstance(col, tuple) else col for col in sp500_filtered.columns]

merged_df = pd.merge(df_selected, sp500_filtered, on="DATE", how="left")
print(merged_df.head())

In [ ]:
merged_df.isna().sum()

In [ ]:
merged_df = merged_df.rename(columns={"SP500_Close" : "sp500"})

In [ ]:
merged_df = merged_df.bfill()

In [ ]:
merged_df["MONTH"] = merged_df["DATE"].dt.to_period("M")
df_monthly = merged_df.groupby("MONTH").last().reset_index()

In [ ]:
df_monthly = df_monthly.drop(columns={"MONTH"})

In [ ]:
df_monthly

In [ ]:
df_monthly.to_csv("data/final.csv")

In [ ]:
df_selected = df_selected.bfill()

In [ ]:
df_weekly = df_selected[df_selected["DATE"].dt.weekday == 4].reset_index(drop=True)

In [ ]:
df_weekly

In [ ]:
df_weekly.isna().sum()

In [ ]:
returns_df = df_weekly.drop(columns=['DATE']).pct_change()

In [ ]:
returns_df.insert(0, 'DATE', df_weekly['DATE'])

In [ ]:
print(returns_df)

In [ ]:
returns = returns_df.iloc[1:]
print(returns)

In [ ]:
if 'ID' in returns.columns:
    returns = returns.drop(columns=['ID'])
    
returns['DATE'] = pd.to_datetime(returns['DATE'])
returns = returns.set_index('DATE')

In [ ]:
returns.to_csv("final3.csv")